# Multipath NxN (Per-device Object IDs)

Each Rx/Tx can use a different scene object for path extraction (e.g., `RX_OBJECT_IDS=["car","uav"]`).

In [1]:
import numpy as np
from pathlib import Path
from sionna.rt import load_scene

from utils import (
    get_object_vertices,
    prepare_path_data,
    make_fixed_path_data,
    setup_tx_rx,
    solve_paths_for_frame,
    plot_path_from_indices,
)

np.set_printoptions(precision=3, suppress=True)


In [ ]:
# 1) Config
XML_PATH = Path('/data/hw/sionna/workspace/scenes/etri_0226/etri_260226_fixed.xml')
MERGE_SHAPES = False
DELTA_T = 0.2
TX_POWER_DBM = 43.0

RX_NAMES = ['rx1', 'rx2']
TX_NAMES = ['tx1', 'tx2']

RX_OBJECT_IDS = ['car', 'uav']
TX_OBJECT_IDS = ['car', 'car']

RX_FIXED = [False, False]
RX_FIXED_POS = [[80.0, -10.0, 1.5], [60.0, 0.0, 1.5]]
TX_FIXED = [True, False]
TX_FIXED_POS = [[100.0, 20.0, 30.0], [120.0, -20.0, 30.0]]

RX_PATH_INDICES_LIST = [[1, 14, 0, 12, 10, 8, 5, 4], [3, 2, 5]]
RX_Z_OFFSET_LIST = [1.5, 20.0]
RX_SPEED_MS_LIST = [100.0/3.6, 60.0/3.6]

TX_PATH_INDICES_LIST = [[0, 0], [4, 5, 8, 10, 12, 0, 14, 1]]
TX_Z_OFFSET_LIST = [20.0, 20.0]
TX_SPEED_MS_LIST = [0.0, 80.0/3.6]

SOLVER_KWARGS = dict(
    max_depth=3,
    samples_per_src=100000,
    diffuse_reflection=True,
    diffraction=True,
    synthetic_array=True,
)


In [7]:
# 2) Sanity checks
nrx = len(RX_NAMES)
ntx = len(TX_NAMES)
assert len(RX_OBJECT_IDS) == nrx
assert len(RX_FIXED) == nrx and len(RX_FIXED_POS) == nrx
assert len(RX_PATH_INDICES_LIST) == nrx
assert len(RX_Z_OFFSET_LIST) == nrx and len(RX_SPEED_MS_LIST) == nrx
assert len(TX_OBJECT_IDS) == ntx
assert len(TX_FIXED) == ntx and len(TX_FIXED_POS) == ntx
assert len(TX_PATH_INDICES_LIST) == ntx
assert len(TX_Z_OFFSET_LIST) == ntx and len(TX_SPEED_MS_LIST) == ntx
print('config check passed | nrx=', nrx, 'ntx=', ntx)


config check passed | nrx= 2 ntx= 2


In [8]:
# 3) Load scene and cache per-object vertices
scene = load_scene(str(XML_PATH), merge_shapes=MERGE_SHAPES)
vertex_cache = {}
for obj_id in set(RX_OBJECT_IDS + TX_OBJECT_IDS):
    vertex_cache[obj_id] = get_object_vertices(scene, obj_id, round_decimals=3)
    print(obj_id, '->', vertex_cache[obj_id].shape)


car -> (16, 3)
uav -> (6, 3)


In [13]:
# 4) Build per-device path_data
rx_path_data_list = []
for i, name in enumerate(RX_NAMES):
    if RX_FIXED[i]:
        pd = make_fixed_path_data(RX_FIXED_POS[i], delta_t=DELTA_T)
        #print(f"{name}: fixed at {RX_FIXED_POS[i]}")
    else:
        positions = vertex_cache[RX_OBJECT_IDS[i]]
        idx = np.asarray(RX_PATH_INDICES_LIST[i], dtype=np.int32)
        if np.any(idx < 0) or np.any(idx >= len(positions)):
            raise ValueError(f"RX idx out of range for {name} ({RX_OBJECT_IDS[i]})")
        pd = prepare_path_data(positions, idx, z_offset=RX_Z_OFFSET_LIST[i], speed_ms=RX_SPEED_MS_LIST[i], delta_t=DELTA_T)
        #print(f"{name}: object={RX_OBJECT_IDS[i]}, frames={len(pd['frame_times'])}")
        #plot_path_from_indices(positions, idx, figsize=(8, 6))
    rx_path_data_list.append(pd)

tx_path_data_list = []
for i, name in enumerate(TX_NAMES):
    if TX_FIXED[i]:
        pd = make_fixed_path_data(TX_FIXED_POS[i], delta_t=DELTA_T)
        #print(f"{name}: fixed at {TX_FIXED_POS[i]}")
    else:
        positions = vertex_cache[TX_OBJECT_IDS[i]]
        idx = np.asarray(TX_PATH_INDICES_LIST[i], dtype=np.int32)
        if np.any(idx < 0) or np.any(idx >= len(positions)):
            raise ValueError(f"TX idx out of range for {name} ({TX_OBJECT_IDS[i]})")
        pd = prepare_path_data(positions, idx, z_offset=TX_Z_OFFSET_LIST[i], speed_ms=TX_SPEED_MS_LIST[i], delta_t=DELTA_T)
        #print(f"{name}: object={TX_OBJECT_IDS[i]}, frames={len(pd['frame_times'])}")
        #plot_path_from_indices(positions, idx, figsize=(8, 6))
    tx_path_data_list.append(pd)


In [14]:
# 5) Setup devices
tx_start_positions = [pd['waypoints'][0].tolist() for pd in tx_path_data_list]
rx_start_positions = [pd['waypoints'][0].tolist() for pd in rx_path_data_list]
tx_names, rx_names, solver = setup_tx_rx(
    scene=scene,
    tx_positions=tx_start_positions,
    rx_positions=rx_start_positions,
    tx_names=TX_NAMES,
    rx_names=RX_NAMES,
    tx_power_dbm=TX_POWER_DBM,
)
print('tx_names=', tx_names)
print('rx_names=', rx_names)


tx_names= ['tx1', 'tx2']
rx_names= ['rx1', 'rx2']


In [15]:
# 6) Solve one frame
FRAME_IDX = 0
paths, rx_pos, rx_vel, tx_pos, tx_vel = solve_paths_for_frame(
    scene=scene,
    solver=solver,
    rx_names=rx_names,
    rx_path_data=rx_path_data_list,
    tx_names=tx_names,
    tx_path_data=tx_path_data_list,
    frame_idx=FRAME_IDX,
    tx_look_at_rx=True,
    tx_look_at_rx_idx=0,
    **SOLVER_KWARGS,
)
print('rx_pos shape:', rx_pos.shape, '| tx_pos shape:', tx_pos.shape)
for i, n in enumerate(rx_names):
    print(f"{n}: pos={rx_pos[i]}, vel={rx_vel[i]}")
for i, n in enumerate(tx_names):
    print(f"{n}: pos={tx_pos[i]}, vel={tx_vel[i]}")
scene.preview(paths=paths, show_devices=True, resolution=[900, 600])


rx_pos shape: (2, 3) | tx_pos shape: (2, 3)
rx1: pos=[ -97.653 -168.273    1.8  ], vel=[23.729 14.442  0.   ]
rx2: pos=[-277.843   83.204   20.3  ], vel=[16.666  0.179  0.   ]
tx1: pos=[100.  20.  30.], vel=[0. 0. 0.]
tx2: pos=[-57.508  55.699  20.3  ], vel=[ -0.398 -22.219   0.   ]
